In [ ]:
import os
import time
import csv
import shutil
import subprocess
from typing import List, Dict, Optional

def _check_tool(cmd: str):
    if shutil.which(cmd) is None:
        raise RuntimeError(f'"{cmd}" not found on PATH. Please install it first.')

def _human(n: int) -> str:
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024 or u == 'TB':
            return f'{n:.2f} {u}' if u != 'B' else f'{n} {u}'
        n /= 1024
    return f'{n:.2f} TB'

In [ ]:
def compress_bpg(
    input_path: str,
    output_path: str,
    qp: int,
    fmt: str = '444',       # 420 / 422 / 444
    bit_depth: int = 8,     # 8 or 10
    encoder_extras: Optional[List[str]] = None,
) -> Dict:
    '''
    Lossy compress an image using bpgenc with a given QP.
    Returns metrics dict.
    '''
    _check_tool('bpgenc')
    if not os.path.isfile(input_path):
        raise FileNotFoundError(input_path)
    os.makedirs(os.path.dirname(os.path.abspath(output_path)) or '.', exist_ok=True)

    # bpgenc 常用参数：
    # -q <qp>        : 量化参数（0~51，值越小质量越高、码率越大；默认29）
    # -f <fmt>       : 色度格式（420/422/444）
    # -b <bit_depth> : 位深（8 或 10）
    # -m <1..9>      : 编码速度/压缩效率权衡（数字越大越慢、压缩略更优），可选
    cmd = ['bpgenc', '-q', str(qp), '-f', fmt, '-b', str(bit_depth), input_path, '-o', output_path]
    if encoder_extras:
        cmd[1:1] = encoder_extras  # 在 -q 前插入额外参数

    t0 = time.perf_counter()
    subprocess.run(cmd, check=True)
    t1 = time.perf_counter()

    in_size = os.path.getsize(input_path)
    out_size = os.path.getsize(output_path)
    ratio = out_size / in_size if in_size > 0 else float('inf')
    

    return {
        'qp': qp,
        'fmt': fmt,
        'bit_depth': bit_depth,
        'input': input_path,
        'output': output_path,
        'input_size': in_size,
        'output_size': out_size,
        'ratio': ratio,
        'time_s': (t1 - t0),
    }

In [ ]:
def decompress_bpg(input_path: str, output_path: str) -> bool:
    _check_tool('bpgdec')
    os.makedirs(os.path.dirname(os.path.abspath(output_path)) or '.', exist_ok=True)
    cmd = ['bpgdec', '-o', output_path, input_path]
    try:
        subprocess.run(cmd, check=True)
        return True
    except Exception as e:
        print(f'Error decompressing {input_path}: {e}')
        return False

In [ ]:
def benchmark_bpg_multi_qp(
    input_path: str,
    out_dir: str,
    qps: List[int],
    fmt: str = '444',
    bit_depth: int = 8,
    encoder_extras: Optional[List[str]] = None,
    csv_path: Optional[str] = None,
) -> List[Dict]:
    '''
    For a list of QPs, run lossy BPG compression and return metrics list.
    Optionally save results to a CSV.
    '''
    os.makedirs(out_dir, exist_ok=True)
    results: List[Dict] = []

    base = os.path.splitext(os.path.basename(input_path))[0]
    for qp in qps:
        out_file = os.path.join(out_dir, f'{base}_qp{qp}.bpg')
        try:
            res = compress_bpg(
                input_path=input_path,
                output_path=out_file,
                qp=qp,
                fmt=fmt,
                bit_depth=bit_depth,
                encoder_extras=encoder_extras,
            )
            results.append(res)
            print(
                f'QP={qp:2d} | out={_human(res['output_size'])} | '
                f'ratio={res['ratio']:.4f} | reduction={res['reduction_pct']:.2f}% | '
                f'time={res['time_s']:.4f}s'
            )
        except Exception as e:
            print(f'[QP={qp}] failed: {e}')

    if csv_path:
        fieldnames = [
            'qp', 'fmt', 'bit_depth', 'input', 'output',
            'input_size', 'output_size', 'ratio', 'reduction_pct', 'time_s'
        ]
        with open(csv_path, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(results)
        print(f'Saved CSV: {csv_path}')

    return results


In [ ]:
import os

dataset = 'SSVTP'
in_dir  = f'/data/ssd/zhaoy/datasets/{dataset}/dataset-comp/test/image'
out_dir = f'/data/ssd/zhaoy/datasets/{dataset}/compressed/bpg/lossy'
os.makedirs(out_dir, exist_ok=True)

qps = [12, 18, 24, 40, 36, 42, 48]

all_results = []
for file in os.listdir(in_dir):
    in_path = os.path.join(in_dir, file)
    results = benchmark_bpg_multi_qp(
            input_path=in_path,
            out_dir=out_dir,
            qps=qps,
            fmt='444',         # 可改 420/422 以进一步降码率
            bit_depth=8,       # 若源是10 bit且你想保留，改 10
            encoder_extras=None,
            csv_path=None
        )
    all_results.extend(results)
    
stat_dir  = '/home/zhaoy/TaCo-Bench/lossy/image/statistics'
os.makedirs(stat_dir, exist_ok=True)

stat_path = os.path.join(stat_dir, f'bpg_{dataset}.csv')
fieldnames = ['qp', 'fmt', 'bit_depth', 'input', 'output', 'input_size', 'output_size', 'ratio', 'reduction_pct', 'time_s']
with open(stat_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(all_results)
print(f'saved csv: {stat_path}.')

[QP=12] failed: 'reduction_pct'
[QP=18] failed: 'reduction_pct'
[QP=24] failed: 'reduction_pct'
[QP=40] failed: 'reduction_pct'
[QP=36] failed: 'reduction_pct'
[QP=42] failed: 'reduction_pct'
[QP=48] failed: 'reduction_pct'
[QP=12] failed: 'reduction_pct'
[QP=18] failed: 'reduction_pct'
[QP=24] failed: 'reduction_pct'
[QP=40] failed: 'reduction_pct'
[QP=36] failed: 'reduction_pct'
[QP=42] failed: 'reduction_pct'
[QP=48] failed: 'reduction_pct'
[QP=12] failed: 'reduction_pct'
[QP=18] failed: 'reduction_pct'
[QP=24] failed: 'reduction_pct'
[QP=40] failed: 'reduction_pct'
[QP=36] failed: 'reduction_pct'
[QP=42] failed: 'reduction_pct'
[QP=48] failed: 'reduction_pct'
[QP=12] failed: 'reduction_pct'
[QP=18] failed: 'reduction_pct'
[QP=24] failed: 'reduction_pct'
[QP=40] failed: 'reduction_pct'
[QP=36] failed: 'reduction_pct'
[QP=42] failed: 'reduction_pct'
[QP=48] failed: 'reduction_pct'
[QP=12] failed: 'reduction_pct'
[QP=18] failed: 'reduction_pct'
[QP=24] failed: 'reduction_pct'
[QP=40] 